# Step 3 — The Block-Window Optimizer

This is the actual "AI planning" component — different from Step 2's model, which only *predicts*. This notebook *decides*: given a maintenance job, what's the best time to schedule it?

**Approach:** for every candidate start hour (0–23), compute a **disruption score** = (real trains/hour at that section+hour) × (planned duration + predicted overrun from the Step 2 model). Lower score = better. Pick the hour that minimizes it, then assign a specific asset.

**Inputs required (same folder):** `03_section_traffic_derived.csv`, `04_asset_register.csv`, `06_daily_asset_availability.csv`, `07_model_training_table.csv`, `overrun_model.joblib`, `model_feature_columns.joblib`


In [1]:
import pandas as pd
import numpy as np
import joblib

traffic = pd.read_csv("03_section_traffic_derived.csv")
assets = pd.read_csv("04_asset_register.csv")
avail = pd.read_csv("06_daily_asset_availability.csv", parse_dates=["date"])
train_table = pd.read_csv("07_model_training_table.csv", parse_dates=["date"])
model = joblib.load("overrun_model.joblib")
feature_columns = joblib.load("model_feature_columns.joblib")

latest_section_rate = (train_table.sort_values("date")
                        .groupby("section_id")["section_historical_overrun_rate"].last())
global_fallback_rate = train_table["overrun_flag"].mean()

avail_with_type = avail.merge(assets[["asset_id", "asset_type"]], on="asset_id", how="left")
overall_type_avail = (avail_with_type.groupby("asset_type")["available"]
                       .apply(lambda s: (s == "YES").mean()))


## Build one candidate-hour's feature row

This must exactly mirror the columns used to train the Step 2 model — same categorical encoding, same column order (`feature_columns`).


In [2]:
def build_feature_row(section_id, hour, maintenance_type, required_asset_type,
                       priority, weather, planned_duration_min, date):
    row = traffic[(traffic.section_id == section_id) & (traffic.hour_of_day == hour)]
    trains_count = float(row["trains_count"].iloc[0]) if len(row) else 0.0
    traffic_level = row["traffic_level"].iloc[0] if len(row) else "LOW"

    assets_here = len(assets[(assets.home_section_id == section_id) &
                              (assets.asset_type == required_asset_type)])

    day_records = avail_with_type[(avail_with_type.date == pd.Timestamp(date)) &
                                   (avail_with_type.asset_type == required_asset_type)]
    if len(day_records):
        avail_pct = (day_records["available"] == "YES").mean()
        had_record = 1
    else:
        avail_pct = overall_type_avail.get(required_asset_type, 0.7)
        had_record = 0

    section_rate = latest_section_rate.get(section_id, global_fallback_rate)
    d = pd.Timestamp(date)
    raw = pd.DataFrame([{
        "day_of_week": d.day_name(), "maintenance_type": maintenance_type,
        "required_asset_type": required_asset_type, "priority": priority,
        "weather": weather, "traffic_level": traffic_level,
        "month": d.month, "is_weekend": int(d.day_name() in ["Saturday", "Sunday"]),
        "start_hour": hour, "trains_count": trains_count,
        "assets_of_type_in_section": assets_here,
        "asset_type_availability_pct": avail_pct, "had_availability_record": had_record,
        "section_historical_overrun_rate": section_rate,
        "planned_duration_min": planned_duration_min,
    }])
    CATEGORICAL = ["day_of_week", "maintenance_type", "required_asset_type", "priority", "weather", "traffic_level"]
    encoded = pd.get_dummies(raw, columns=CATEGORICAL).reindex(columns=feature_columns, fill_value=0)
    return encoded, trains_count, traffic_level, avail_pct


## The optimizer: try every hour, score it, pick the best

`disruption_score = trains_count × expected_total_duration` — a simple, explainable proxy for "how many train-minutes of disruption does this cause." You can swap this formula for a fancier one later (e.g. weighting by train priority/passenger count), but this is a legitimate, defensible first version.


In [3]:
def recommend_block_window(section_id, maintenance_type, required_asset_type,
                            priority, weather, planned_duration_min, date,
                            candidate_hours=range(24)):
    results = []
    for hour in candidate_hours:
        X_row, trains_count, traffic_level, avail_pct = build_feature_row(
            section_id, hour, maintenance_type, required_asset_type,
            priority, weather, planned_duration_min, date)
        predicted_overrun = float(model.predict(X_row)[0])
        expected_total_duration = planned_duration_min + max(predicted_overrun, 0)
        disruption_score = trains_count * expected_total_duration
        results.append(dict(
            start_hour=hour, trains_count=trains_count, traffic_level=traffic_level,
            predicted_overrun_min=round(predicted_overrun, 1),
            expected_total_duration_min=round(expected_total_duration, 1),
            asset_type_availability_pct=round(float(avail_pct), 2),
            disruption_score=round(disruption_score, 1),
        ))
    ranked = pd.DataFrame(results).sort_values("disruption_score").reset_index(drop=True)
    best = ranked.iloc[0]

    candidates = assets[assets.asset_type == required_asset_type].copy()
    candidates["home_match"] = (candidates.home_section_id == section_id).astype(int)
    candidates = candidates.sort_values(["home_match", "status"], ascending=[False, True])
    selected_asset = candidates.iloc[0] if len(candidates) else None

    print("="*50)
    print("BLOCK RECOMMENDATION")
    print("="*50)
    print(f"Section:              {section_id}")
    print(f"Maintenance type:     {maintenance_type}")
    print(f"Date:                 {date}")
    print(f"Recommended start:    {int(best.start_hour):02d}:00")
    print(f"Expected duration:    {best.expected_total_duration_min} min "
          f"(planned {planned_duration_min}, predicted overrun {best.predicted_overrun_min})")
    print(f"Required asset:       {required_asset_type}")
    print(f"Selected asset:       {selected_asset['asset_id'] if selected_asset is not None else 'NONE AVAILABLE'} "
          f"(status: {selected_asset['status'] if selected_asset is not None else '-'})")
    print(f"Traffic at that hour: {best.traffic_level} ({int(best.trains_count)} trains/hr historically)")
    print(f"Disruption score:     {best.disruption_score}  (lower is better)")
    print("-"*50)
    print("Top 5 candidate hours, ranked:")
    print(ranked.head(5).to_string(index=False))
    return ranked, selected_asset


## Try it on a real busy section

`CLA-MTN` was one of the busiest real sections found in Step 1 (393 scheduled trains) — a good stress test.


In [4]:
ranked, asset = recommend_block_window(
    section_id="CLA-MTN",
    maintenance_type="Track Repair",
    required_asset_type="Tamping Machine",
    priority="High",
    weather="Clear",
    planned_duration_min=180,
    date="2026-03-15",
)


BLOCK RECOMMENDATION
Section:              CLA-MTN
Maintenance type:     Track Repair
Date:                 2026-03-15
Recommended start:    03:00
Expected duration:    195.2 min (planned 180, predicted overrun 15.2)
Required asset:       Tamping Machine
Selected asset:       TA005 (status: Available)
Traffic at that hour: LOW (0 trains/hr historically)
Disruption score:     0.0  (lower is better)
--------------------------------------------------
Top 5 candidate hours, ranked:
 start_hour  trains_count traffic_level  predicted_overrun_min  expected_total_duration_min  asset_type_availability_pct  disruption_score
          3           0.0           LOW                   15.2                        195.2                         0.85               0.0
          2           0.0           LOW                   14.6                        194.6                         0.85               0.0
          1           1.0           LOW                   13.4                        193.4         

## What to check here

- The optimizer should recommend a **low-traffic night hour** — if it instead picks a peak hour, something's wrong with the traffic join.
- `disruption_score` should clearly rise for daytime hours in the ranked table — this is your evidence the optimizer is actually responding to real traffic patterns, not guessing randomly.

**Next (Step 4):** compare this optimizer's picks against a dumb baseline (e.g. "always schedule at midnight" or "always use the nearest asset regardless of traffic") across many blocks, to quantify how much disruption it actually saves.
